In [2]:
from policy.power.proportional import ProportionalPower
from models.ev import EV
from models.pile import ChargingPile

In [3]:
# Test 1: non-overloaded case
policy = ProportionalPower()

ev1 = EV(id=1, c_b=25, s_i=0.2, s_f=0.8, arrival_time=0.0, s_th=0.5)   # p_req ~= 20
ev2 = EV(id=2, c_b=33, s_i=0.2, s_f=0.8, arrival_time=0.0, s_th=0.5)   # p_req ~= 30
pile = ChargingPile(id=0, n=2, num_bricks=10, p_brick=10)

pile.connect_ev(ev1)
pile.connect_ev(ev2)
assignments = policy.update_power(pile)

print("ev_bricks:", pile.ev_bricks)
print("assignments:", [(ev.id, p) for ev, p in assignments])

# Expected: ceil(25/10)=3 and ceil(33/10)=4
assert pile.ev_bricks == [3, 4]
assert assignments[0][1] == 30
assert assignments[1][1] == 40

ev_bricks: [3, 4]
assignments: [(1, 30), (2, 40)]


In [4]:
# Test 2: non-overloaded case
policy = ProportionalPower()

ev1 = EV(id=1, c_b=5, s_i=0.2, s_f=0.8, arrival_time=0.0, s_th=0.5)   # p_req ~= 20
ev2 = EV(id=2, c_b=35, s_i=0.2, s_f=0.8, arrival_time=0.0, s_th=0.5)   # p_req ~= 30
pile = ChargingPile(id=0, n=2, num_bricks=10, p_brick=10)

pile.connect_ev(ev1)
pile.connect_ev(ev2)
assignments = policy.update_power(pile)

print("ev_bricks:", pile.ev_bricks)
print("assignments:", [(ev.id, p) for ev, p in assignments])

# Expected: ceil(5/10)=1 and ceil(35/10)=4
assert pile.ev_bricks == [1, 4]
assert assignments[0][1] == 10
assert assignments[1][1] == 40

ev_bricks: [1, 4]
assignments: [(1, 10), (2, 40)]


In [10]:
# Test 3: overloaded invariants
policy = ProportionalPower()

ev1 = EV(id=1, c_b=300, s_i=0.2, s_f=0.8, arrival_time=0.0, s_th=0.5)  # p_req ~= 100
ev2 = EV(id=2, c_b=500, s_i=0.2, s_f=0.8, arrival_time=0.0, s_th=0.5)  # p_req ~= 500
pile = ChargingPile(id=0, n=2, num_bricks=10, p_brick=10)

pile.connect_ev(ev1)
pile.connect_ev(ev2)
assignments = policy.update_power(pile)

print("is_overloaded:", pile.is_overloaded)
print("ev_bricks:", pile.ev_bricks, "sum:", sum(pile.ev_bricks), "num_bricks:", pile.num_bricks)
print("assignments:", [(ev.id, p) for ev, p in assignments])

# In an overloaded pile, total assigned bricks should never exceed capacity.
assert sum(pile.ev_bricks) <= pile.num_bricks
assert all(b >= 1 for b in pile.ev_bricks)

is_overloaded: True
ev_bricks: [3, 7] sum: 10 num_bricks: 10
assignments: [(1, 30), (2, 70)]


In [ ]:
# Test 4: overloaded invariants
policy = ProportionalPower()

ev1 = EV(id=1, c_b=9, s_i=0.2, s_f=0.8, arrival_time=0.0, s_th=0.5)  # p_req ~= 9
ev2 = EV(id=2, c_b=500, s_i=0.2, s_f=0.8, arrival_time=0.0, s_th=0.5)  # p_req ~= 500
pile = ChargingPile(id=0, n=2, num_bricks=10, p_brick=10)

pile.connect_ev(ev1)
pile.connect_ev(ev2)
assignments = policy.update_power(pile)

print("is_overloaded:", pile.is_overloaded)
print("ev_bricks:", pile.ev_bricks, "sum:", sum(pile.ev_bricks), "num_bricks:", pile.num_bricks)
print("assignments:", [(ev.id, p) for ev, p in assignments])

# In an overloaded pile, total assigned bricks should never exceed capacity.
assert sum(pile.ev_bricks) <= pile.num_bricks
assert all(b >= 1 for b in pile.ev_bricks)

is_overloaded: True
ev_bricks: [1, 9] sum: 10 num_bricks: 10
assignments: [(1, 10), (2, 90)]


In [ ]:
# Test 5: micro distribution case - pile not overloaded
policy = ProportionalPower()

ev1 = EV(id=1, c_b=35, s_i=0.2, s_f=0.8, arrival_time=0.0, s_th=0.5)
ev2 = EV(id=2, c_b=45, s_i=0.2, s_f=0.8, arrival_time=0.0, s_th=0.5)
pile = ChargingPile(id=0, n=2, num_bricks=10, p_brick=10)

pile.connect_ev(ev1)
pile.connect_ev(ev2)
assignments = policy.update_power(pile)
print(f"Since pile.is_overloaded = {pile.is_overloaded}, micro distribution shouldn't change anything.")
assignments_new_ev1 = policy.update_power(pile, ev1)
assignments_new_ev2 = policy.update_power(pile, ev2)
assert all(old[0].n_bricks == new[0].n_bricks for (old, new) in zip(assignments, assignments_new_ev1))
assert all(old[0].n_bricks == new[0].n_bricks for (old, new) in zip(assignments, assignments_new_ev2))
print("Test case passed!")


Since pile.is_overloaded = False, micro distribution shouldn't change anything.
Test case passed!


In [ ]:
# Test 6: micro distribution case - pile overloaded
policy = ProportionalPower()

ev1 = EV(id=1, c_b=120, s_i=0.2, s_f=0.8, arrival_time=0.0, s_th=0.5)
ev2 = EV(id=2, c_b=240, s_i=0.2, s_f=0.8, arrival_time=0.0, s_th=0.5)
pile = ChargingPile(id=0, n=2, num_bricks=10, p_brick=10)

pile.connect_ev(ev1)
pile.connect_ev(ev2)
assignments = policy.update_power(pile)
print("ev_bricks:", pile.ev_bricks, "sum:", sum(pile.ev_bricks), "num_bricks:", pile.num_bricks)
print("assignments:", [(ev.id, p) for ev, p in assignments])
print("remaining power reqs (id, kW):", [(idx+1, p) for idx, p in enumerate(pile.remaining_reqs)])
# Unassigning one brick ev1, and then redistributing that to the
# largest remaining request, which is ev2.
assignments_new_ev1 = policy.update_power(pile, ev1, kind='micro')
print("assignments new:", [(ev.id, p) for ev, p in assignments_new_ev1])
ev1_passed = assignments[0][1] - pile.p_brick == assignments_new_ev1[0][1]
ev2_passed = assignments[1][1] + pile.p_brick == assignments_new_ev1[1][1]
assert ev1_passed and ev2_passed
print("Test case passed!")


ev_bricks: [3, 7] sum: 10 num_bricks: 10
assignments: [(1, 30), (2, 70)]
remaining power reqs (id, kW): [(1, 90.0), (2, 170.0)]
assignments new: [(1, 20), (2, 80)]
assignments new: [(1, 20), (2, 80)]


In [61]:
# Test 6: micro distribution case - pile overloaded
policy = ProportionalPower()

ev1 = EV(id=1, c_b=120, s_i=0.2, s_f=0.8, arrival_time=0.0, s_th=0.5)
ev2 = EV(id=2, c_b=240, s_i=0.2, s_f=0.8, arrival_time=0.0, s_th=0.5)
pile = ChargingPile(id=0, n=2, num_bricks=10, p_brick=10)

pile.connect_ev(ev1)
pile.connect_ev(ev2)
assignments = policy.update_power(pile)
print("ev_bricks:", pile.ev_bricks, "sum:", sum(pile.ev_bricks), "num_bricks:", pile.num_bricks)
print("assignments:", [(ev.id, p) for ev, p in assignments])
print("remaining power reqs (id, kW):", [(idx+1, p) for idx, p in enumerate(pile.remaining_reqs)])
# Unassigning one brick ev2, and then redistributing that to the
# largest remaining request, which is ev2, so the new and old assignment should be the same.
assignments_new_ev2 = policy.update_power(pile, ev2, kind='micro')
print("assignments new:", [(ev.id, p) for ev, p in assignments_new_ev2])
ev1_passed = assignments[0][1] == assignments_new_ev2[0][1]
ev2_passed = assignments[1][1] == assignments_new_ev2[1][1]
assert ev1_passed and ev2_passed
print("Test case passed!")


ev_bricks: [3, 7] sum: 10 num_bricks: 10
assignments: [(1, 30), (2, 70)]
remaining power reqs (id, kW): [(1, 90.0), (2, 170.0)]
assignments new: [(1, 30), (2, 70)]
Test case passed!
